# Section 5.1: periodic Fourier model on the two-dimensional torus

Reversible diffusion on the torus $\mathbb T^2$ with uniform invariant law and generator
$\mathcal A f=\nabla\cdot(a\nabla f)$, $a(x)=1+\tfrac{\beta}{2}\cos((1,1)\cdot x)+\tfrac{\beta}{2}\cos((1,-1)\cdot x)$, $\beta=1/2$ (the product $1+\beta\cos x_1\cos x_2$ of the cosines of the two coordinates), that is, the SDE
$dX=\nabla a(X)\,dt+\sqrt{2a(X)}\,dW$. Dictionary: the $N_K=(2K+1)^2$ tensor trigonometric modes with
$\max(|k_1|,|k_2|)\le K$ in the real orthonormal form $1,\sqrt2\cos(k\cdot x),\sqrt2\sin(k\cdot x)$.
Observable: $\hat f_k\propto(1+|k|^2)^{-4}$, normalized to $\|f\|=1$ ($f\in H^\ell(\mathbb T^2)$ for $\ell<7$).

Since $\mathcal A e_k=-|k|^2e_k-\tfrac{\beta}{4}\sum_{s\in\{(\pm1,\pm1)\}}k\cdot(k+s)\,e_{k+s}$, the Galerkin matrices are explicit.
For $T=1$ and $\Delta t=T/n$ with $n$ the smallest integer such that $\Delta t\,\|\mathcal A_K\|\le1.5$, the notebook reproduces

* Table 4: the errors $\|\mathcal Q^n\Pi_Kf-\mathcal K^Tf\|$ of SDMD-1, SDMD-R, SDMD-2 and the Galerkin error with exact matrices, and the
  errors of the three schemes and of the exponential $e^{T\widehat{\mathcal A}}$ with empirical matrices from $m=10^4$ i.i.d. uniform
  samples (seed 20261029);
* the rows for $\mathbb T^2$ of Table 5 (computing times) and the data for the right panel of Figure 1
  (`results_torus2d.json`, read by `5_1_fourier_accuracy_cost.ipynb`).

The reference solution is the Fourier--Galerkin semigroup with cutoff $64$ ($N=16641$), which agrees with the cutoffs $48$ and
$80$ to $5\times10^{-13}$. Runtime: about three minutes.

In [1]:
import json
import time

import numpy as np
import scipy.linalg as sla
import scipy.sparse as sp
from scipy.sparse.linalg import expm_multiply

SEED = 20261029
rng = np.random.default_rng(SEED)
beta, T = 0.5, 1.0
S = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]])


def modes(K):
    r = np.arange(-K, K + 1)
    k1, k2 = np.meshgrid(r, r, indexing="ij")
    return np.stack([k1.ravel(), k2.ravel()], axis=1)


def idx_of(K, k1, k2):
    return (k1 + K) * (2 * K + 1) + (k2 + K)


def A_sparse(K):
    """Galerkin matrix of A on V_K in the complex exponential basis (real symmetric)."""
    km = modes(K)
    N = len(km)
    rows, cols, vals = [], [], []
    for j, (k1, k2) in enumerate(km):
        rows.append(j); cols.append(j); vals.append(-float(k1 * k1 + k2 * k2))
        for s in S:
            l1, l2 = k1 + s[0], k2 + s[1]
            if max(abs(l1), abs(l2)) <= K:
                rows.append(idx_of(K, l1, l2)); cols.append(j)
                vals.append(-beta / 4 * float(k1 * l1 + k2 * l2))
    return sp.csr_matrix((vals, (rows, cols)), shape=(N, N))


def sub_index(Kbig, K):
    km = modes(K)
    return idx_of(Kbig, km[:, 0], km[:, 1])


def exact_matrices(K):
    """A_K and B_K = Pi_K A^2|_{V_K}; A e_k has support in V_{K+1}, so B_K = (A_{K+1}^2)[V_K, V_K]."""
    A = A_sparse(K).toarray()
    A1 = A_sparse(K + 1)
    sub = sub_index(K + 1, K)
    B = (A1 @ A1).toarray()[np.ix_(sub, sub)]
    return A, B


def f_coeffs(Kref):
    km = modes(Kref)
    c = (1.0 + (km ** 2).sum(1)) ** (-4.0)
    return c / np.linalg.norm(c)


def reference(Kref):
    f = f_coeffs(Kref)
    return f, expm_multiply(T * A_sparse(Kref), f)


def embed(vec, K, Kref):
    out = np.zeros((2 * Kref + 1) ** 2, dtype=np.result_type(vec, float))
    out[sub_index(Kref, K)] = vec
    return out


def step_count(norm_A):
    """Number of steps n and step dt = T/n, with n the smallest integer such that dt * norm_A <= 1.5."""
    n = int(np.ceil(T * norm_A / 1.5))
    return n, T / n


def one_step_maps(A, B, dt):
    I = np.eye(len(A))
    return {"sdmd1": I + dt * A, "sdmdR": I + dt * A + 0.5 * dt * dt * (A @ A), "sdmd2": I + dt * A + 0.5 * dt * dt * B}


def advance(Q, n, v):
    for _ in range(n):
        v = Q @ v
    return v


def spectral_norm_sym(M):
    return float(np.abs(sla.eigvalsh(M)).max())


# real orthonormal basis: 1, sqrt2 cos(k.x), sqrt2 sin(k.x), k in a half lattice
def half_lattice(K):
    km = modes(K)
    return np.array([(k1, k2) for k1, k2 in km if (k1 > 0) or (k1 == 0 and k2 > 0)])


def real_from_complex_matrix(K):
    """Unitary U with phi_real = E_complex @ U."""
    N = (2 * K + 1) ** 2
    H = half_lattice(K)
    U = np.zeros((N, N), dtype=complex)
    U[idx_of(K, 0, 0), 0] = 1.0
    for a, (k1, k2) in enumerate(H):
        ip, im = idx_of(K, k1, k2), idx_of(K, -k1, -k2)
        U[ip, 1 + a] = 1 / np.sqrt(2); U[im, 1 + a] = 1 / np.sqrt(2)
        U[ip, 1 + len(H) + a] = 1 / (np.sqrt(2) * 1j); U[im, 1 + len(H) + a] = -1 / (np.sqrt(2) * 1j)
    return U


def to_real_basis(M, U):
    R = U.conj().T @ M @ U
    assert np.abs(R.imag).max() < 1e-9 * (1 + np.abs(R).max())
    return R.real


def coefficient_fields(x):
    """a, grad a, the second derivatives a_11, a_12, a_22, and Delta a at the points x (m x 2)."""
    c1, s1, c2, s2 = np.cos(x[:, 0]), np.sin(x[:, 0]), np.cos(x[:, 1]), np.sin(x[:, 1])
    a = 1.0 + beta * c1 * c2
    grad = np.stack([-beta * s1 * c2, -beta * c1 * s2], axis=1)
    return a, grad, (-beta * c1 * c2, beta * s1 * s2, -beta * c1 * c2), -2.0 * beta * c1 * c2


def real_arrays(x, K, with_second):
    """Values of the real dictionary, of A psi and (optionally) of A^2 psi at the sample points x (m x 2).

    For e_k = exp(i k.x) the generator A f = a Delta f + grad a . grad f acts pointwise as A e_k = c_k e_k with
    c_k = -a |k|^2 + i k.grad a, and A^2 e_k = (c_k^2 + A c_k + 2 i a k.grad c_k) e_k. Real and imaginary parts
    give the images of sqrt2 cos(k.x) and sqrt2 sin(k.x)."""
    H = half_lattice(K)
    nH = len(H)
    k1, k2 = H[:, 0].astype(float), H[:, 1].astype(float)
    k_sq = k1**2 + k2**2
    a, grad, (a11, a12, a22), lap = coefficient_fields(x)
    a, g1, g2 = a[:, None], grad[:, :1], grad[:, 1:]
    E = np.exp(1j * (x @ H.T))                         # m x nH
    kg = g1 * k1 + g2 * k2                             # k . grad a
    c = -a * k_sq + 1j * kg

    def real_image(W, first):
        out = np.empty((len(x), 1 + 2 * nH))
        out[:, 0] = first
        out[:, 1:1 + nH] = np.sqrt(2) * W.real
        out[:, 1 + nH:] = np.sqrt(2) * W.imag
        return out

    Psi = real_image(E, 1.0)
    APsi = real_image(c * E, 0.0)
    A2Psi = None
    if with_second:
        # grad c_k = -|k|^2 grad a + i Hess(a) k,  Delta c_k = -|k|^2 Delta a + i k.grad(Delta a),  grad(Delta a) = -2 grad a
        gc1 = -g1 * k_sq + 1j * (a11[:, None] * k1 + a12[:, None] * k2)
        gc2 = -g2 * k_sq + 1j * (a12[:, None] * k1 + a22[:, None] * k2)
        Ac = a * (-lap[:, None] * k_sq - 2j * kg) + g1 * gc1 + g2 * gc2
        A2Psi = real_image((c * c + Ac + 2j * a * (k1 * gc1 + k2 * gc2)) * E, 0.0)
    return Psi, APsi, A2Psi


def empirical_matrices(x, K, with_second, chunk=10000):
    """Empirical G, A = G^{-1} H and (if with_second) B = G^{-1} J from the samples x, accumulated in chunks."""
    m = len(x); N = (2 * K + 1) ** 2
    G = np.zeros((N, N)); Hm = np.zeros((N, N)); J = np.zeros((N, N)) if with_second else None
    for s in range(0, m, chunk):
        Psi, APsi, A2Psi = real_arrays(x[s:s + chunk], K, with_second)
        G += Psi.T @ Psi; Hm += Psi.T @ APsi
        if with_second:
            J += Psi.T @ A2Psi
    G /= m; Hm /= m
    if not with_second:
        return G, np.linalg.solve(G, Hm), None
    AB = np.linalg.solve(G, np.hstack([Hm, J / m]))    # one factorization of G for both right-hand sides
    return G, AB[:, :N], AB[:, N:]


def g(x):
    return f"{x:.2g}"

## Reference solution, and a check of the pointwise formulas

The arrays $\mathcal A\boldsymbol\Psi$ and $\mathcal A^2\boldsymbol\Psi$ are evaluated pointwise in `real_arrays`. They are compared here with the
expansions of $\mathcal Ae_k$ in $\mathcal V_{K+1}$ and of $\mathcal A^2e_k$ in $\mathcal V_{K+2}$ given by the Galerkin matrices.

In [2]:
Kref = 64
f_ref, u_ref = reference(Kref)
for Kc in (48, 80):
    fc, uc = reference(Kc)
    a, b = (u_ref, embed(uc, Kc, Kref)) if Kc < Kref else (embed(u_ref, Kref, Kc), uc)
    print(f"|u_{Kref} - u_{Kc}| = {np.linalg.norm(a - b):.2e}")

Kc = 3
xc = np.random.default_rng(0).uniform(0, 2 * np.pi, size=(9, 2))
Uc = real_from_complex_matrix(Kc)
Psi, APsi, A2Psi = real_arrays(xc, Kc, True)
A1, A2 = A_sparse(Kc + 1), A_sparse(Kc + 2)
C1 = A1[:, sub_index(Kc + 1, Kc)].toarray()            # A e_k expanded in V_{K+1}
C2 = (A2 @ A2)[:, sub_index(Kc + 2, Kc)].toarray()     # A^2 e_k expanded in V_{K+2}
E = lambda Kk: np.exp(1j * (xc @ modes(Kk).T))
print("pointwise A psi, A^2 psi versus the Fourier expansions: max deviations "
      f"{np.abs(APsi - E(Kc + 1) @ C1 @ Uc).max():.1e}, {np.abs(A2Psi - E(Kc + 2) @ C2 @ Uc).max():.1e}")

|u_64 - u_48| = 5.26e-13


|u_64 - u_80| = 5.27e-13
pointwise A psi, A^2 psi versus the Fourier expansions: max deviations 1.8e-14, 6.3e-13


## Exact Galerkin matrices

In [3]:
exact = []
print(f"{'K':>3} {'N':>5} {'n':>5} {'dt|A_K|':>8} {'||A_K||':>9} {'Theta_K':>9} {'Galerkin':>10} {'SDMD-1':>10} {'SDMD-R':>10} {'SDMD-2':>10} {'max||Q||':>9}")
for K in [2, 4, 8, 16, 32]:
    A, B = exact_matrices(K)
    normA = spectral_norm_sym(A)
    Theta = spectral_norm_sym(B - A @ A)
    n, dt = step_count(normA)
    fK = f_ref[sub_index(Kref, K)]
    Q = one_step_maps(A, B, dt)
    err = {name: float(np.linalg.norm(embed(advance(M, n, fK), K, Kref) - u_ref)) for name, M in Q.items()}
    err["galerkin"] = float(np.linalg.norm(embed(expm_multiply(T * sp.csr_matrix(A), fK), K, Kref) - u_ref))
    normQ = max(spectral_norm_sym(M) for M in Q.values())
    exact.append(dict(K=K, N=len(A), n=n, dt=dt, normA=normA, Theta=Theta, errors=err, normQ=normQ))
    print(f"{K:>3} {len(A):>5} {n:>5} {dt*normA:>8.3f} {g(normA):>9} {g(Theta):>9} {err['galerkin']:>10.2e} {err['sdmd1']:>10.3e} {err['sdmdR']:>10.3e} {err['sdmd2']:>10.3e} {normQ:>9.6f}")

Ks = np.array([e["K"] for e in exact], float)
for name in ["sdmd1", "sdmdR", "sdmd2"]:
    err = np.array([e["errors"][name] for e in exact])
    print(f"fitted slope of {name} against K over K = 8, 16, 32: {np.polyfit(np.log(Ks[-3:]), np.log(err[-3:]), 1)[0]:.2f}")
print("relative difference (SDMD-2 - SDMD-R) / SDMD-R:",
      ", ".join(f"K={e['K']}: {(e['errors']['sdmd2'] - e['errors']['sdmdR']) / e['errors']['sdmdR']:+.1e}" for e in exact))

  K     N     n  dt|A_K|   ||A_K||   Theta_K   Galerkin     SDMD-1     SDMD-R     SDMD-2  max||Q||
  2    25     6    1.340         8       4.9   6.48e-04  4.202e-03  7.471e-04  6.820e-04  1.000000
  4    81    22    1.484        33        65   2.42e-05  1.062e-03  3.052e-05  2.969e-05  1.000000
  8   289    91    1.489   1.4e+02     1e+03   5.92e-08  2.530e-04  1.030e-06  1.030e-06  1.000000


 16  1089   391    1.496   5.9e+02   1.6e+04   8.77e-13  5.870e-05  5.519e-08  5.519e-08  1.000000


 32  4225  1692    1.500   2.5e+03   2.7e+05   5.27e-13  1.355e-05  2.941e-09  2.941e-09  1.000000
fitted slope of sdmd1 against K over K = 8, 16, 32: -2.11
fitted slope of sdmdR against K over K = 8, 16, 32: -4.23
fitted slope of sdmd2 against K over K = 8, 16, 32: -4.23
relative difference (SDMD-2 - SDMD-R) / SDMD-R: K=2: -8.7e-02, K=4: -2.7e-02, K=8: -1.2e-04, K=16: -4.5e-12, K=32: +0.0e+00


## Empirical matrices from $m=10^4$ i.i.d. uniform samples ($\gamma=0$)

In [4]:
empirical = []
print(f"{'K':>3} {'N':>5} {'m':>6} {'reps':>4} {'exp(T Ahat)':>12} {'SDMD-1':>10} {'SDMD-R':>10} {'SDMD-2':>10} {'|Ahat-A|':>9} {'|Bhat-B|':>9} {'cond G':>7}")
for K, m, reps in [(2, 10**4, 10), (4, 10**4, 10), (8, 10**4, 10), (16, 10**4, 5)]:
    A, B = exact_matrices(K)
    U = real_from_complex_matrix(K)
    Ar, Br = to_real_basis(A, U), to_real_basis(B, U)
    n, dt = step_count(spectral_norm_sym(A))
    fK_real = (U.conj().T @ f_ref[sub_index(Kref, K)]).real
    error = lambda v: float(np.linalg.norm(embed(U @ v, K, Kref) - u_ref))
    out = {k: [] for k in ["exp", "sdmd1", "sdmdR", "sdmd2", "xiA", "xiB", "condG"]}
    for r in range(reps):
        x = rng.uniform(0, 2 * np.pi, size=(m, 2))
        G, Ah, Bh = empirical_matrices(x, K, True)
        out["xiA"].append(float(np.linalg.norm(Ah - Ar, 2))); out["xiB"].append(float(np.linalg.norm(Bh - Br, 2)))
        out["condG"].append(float(np.linalg.cond(G)))
        out["exp"].append(error(sla.expm(T * Ah) @ fK_real))
        for name, M in one_step_maps(Ah, Bh, dt).items():
            out[name].append(error(advance(M, n, fK_real)))
    mean = {k: float(np.mean(v)) for k, v in out.items()}
    empirical.append(dict(K=K, N=len(A), m=m, reps=reps, n=n, dt=dt, mean=mean))
    print(f"{K:>3} {len(A):>5} {m:>6} {reps:>4} {mean['exp']:>12.2e} {mean['sdmd1']:>10.3e} {mean['sdmdR']:>10.3e} {mean['sdmd2']:>10.3e} {g(mean['xiA']):>9} {g(mean['xiB']):>9} {g(mean['condG']):>7}")

pop = {e["K"]: e["errors"] for e in exact}
print("mean sampled error / error with exact matrices:")
for e in empirical:
    print(f"  K={e['K']:>2}: " + ", ".join(f"{name} {e['mean'][name] / pop[e['K']][name]:.4f}" for name in ["sdmd1", "sdmdR", "sdmd2"])
          + f", exponential {e['mean']['exp'] / pop[e['K']]['galerkin']:.3g}")

  K     N      m reps  exp(T Ahat)     SDMD-1     SDMD-R     SDMD-2  |Ahat-A|  |Bhat-B|  cond G
  2    25  10000   10     6.55e-04  4.225e-03  7.426e-04  6.850e-04      0.16       3.4     1.2


  4    81  10000   10     2.61e-05  1.062e-03  3.179e-05  3.002e-05      0.96        67     1.5


  8   289  10000   10     9.57e-08  2.530e-04  1.041e-06  1.034e-06       6.6   1.7e+03     2.4


 16  1089  10000    5     3.23e-12  5.870e-05  5.519e-08  5.519e-08        54   5.3e+04     7.8
mean sampled error / error with exact matrices:
  K= 2: sdmd1 1.0056, sdmdR 0.9940, sdmd2 1.0045, exponential 1.01
  K= 4: sdmd1 1.0001, sdmdR 1.0416, sdmd2 1.0111, exponential 1.08
  K= 8: sdmd1 1.0000, sdmdR 1.0101, sdmd2 1.0039, exponential 1.62
  K=16: sdmd1 1.0000, sdmdR 1.0000, sdmd2 1.0000, exponential 3.69


## Table 5 (rows for $\mathbb T^2$): computing times with $m=10^4$ samples

Times are the best of three runs on one further sample, after one run that is not timed. The time for the matrices includes the evaluation of the arrays
$\mathcal A\boldsymbol\Psi$ (and $\mathcal A^2\boldsymbol\Psi$ for SDMD-2) and the solution with $\widehat{\mathbf G}_m$; the time stepping consists of the
one-step matrix and $n$ matrix-vector products; the exponential is `scipy.linalg.expm` applied to $T\widehat{\mathbf A}$.

In [5]:
def timed(fn, *args, reps=3):
    """Best time of reps runs, after one run that is not timed."""
    best, out = np.inf, fn(*args)
    for _ in range(reps):
        t0 = time.perf_counter()
        out = fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best, out


def run_sdmd1(A, dt, n, v):
    return advance(np.eye(len(A)) + dt * A, n, v)


def run_sdmdR(A, dt, n, v):
    return advance(np.eye(len(A)) + dt * A + 0.5 * dt * dt * (A @ A), n, v)


def run_sdmd2(A, B, dt, n, v):
    return advance(np.eye(len(A)) + dt * A + 0.5 * dt * dt * B, n, v)


def run_exp(A, v):
    return sla.expm(T * A) @ v


timing = []
print("computing times in seconds, m = 1e4")
print(f"{'K':>3} {'N':>5} {'n':>5} {'matrices(1,R)':>14} {'matrices(2)':>12} {'ratio':>6} {'step(1)':>9} {'step(R)':>9} {'step(2)':>9} {'exponential':>12}")
emp = {e["K"]: e for e in empirical}
for K, m in [(2, 10**4), (4, 10**4), (8, 10**4), (16, 10**4)]:
    x = rng.uniform(0, 2 * np.pi, size=(m, 2))
    t1, (_, Ah, _) = timed(empirical_matrices, x, K, False)
    t2, (_, Ah2, Bh) = timed(empirical_matrices, x, K, True)
    assert np.allclose(Ah, Ah2)
    n, dt = emp[K]["n"], emp[K]["dt"]
    U = real_from_complex_matrix(K)
    v0 = (U.conj().T @ f_ref[sub_index(Kref, K)]).real
    ts = {"sdmd1": timed(run_sdmd1, Ah, dt, n, v0)[0], "sdmdR": timed(run_sdmdR, Ah, dt, n, v0)[0],
          "sdmd2": timed(run_sdmd2, Ah, Bh, dt, n, v0)[0]}
    te = timed(run_exp, Ah, v0)[0]
    timing.append(dict(K=K, N=(2 * K + 1) ** 2, m=m, n=n, err={k: emp[K]["mean"][k] for k in ["exp", "sdmd1", "sdmdR", "sdmd2"]},
                       t_1R=t1, t_2=t2, ts=ts, t_exp=te))
    print(f"{K:>3} {(2*K+1)**2:>5} {n:>5} {g(t1):>14} {g(t2):>12} {t2/t1:>6.2f} {g(ts['sdmd1']):>9} {g(ts['sdmdR']):>9} {g(ts['sdmd2']):>9} {g(te):>12}")

json.dump(timing, open("results_torus2d.json", "w"), indent=1)
print("wrote results_torus2d.json (errors with m = 1e4 and computing times; data for the right panel of Figure 1)")

computing times in seconds, m = 1e4
  K     N     n  matrices(1,R)  matrices(2)  ratio   step(1)   step(R)   step(2)  exponential
  2    25     6         0.0069        0.013   1.82   7.3e-06   1.1e-05   8.7e-06        3e-05


  4    81    22          0.023        0.048   2.06   3.4e-05   6.1e-05     4e-05      0.00039


  8   289    91           0.12         0.23   1.91   0.00088    0.0019    0.0011        0.015


 16  1089   391           0.92            2   2.20      0.13      0.17      0.15         0.66
wrote results_torus2d.json (errors with m = 1e4 and computing times; data for the right panel of Figure 1)


## LaTeX rows of Tables 4 and 5

In [6]:
def sci(x, sig=3):
    """LaTeX form with sig significant digits."""
    mant, ex = f"{x:.{sig-1}e}".split("e")
    return f"${mant}\\times10^{{{int(ex)}}}$"


def plain(x, sig=2):
    """sig significant digits, in positional notation between 0.001 and 100."""
    if not 1e-3 <= abs(x) < 1e2:
        return sci(x, sig)
    digits = max(sig - 1 - int(np.floor(np.log10(abs(x)))), 0)
    return f"${round(x, digits):.{digits}f}$"


def small(x, floor=1e-12):
    return sci(x, 2) if x >= floor else "$<10^{-12}$"


print("% Table 4, exact matrices")
for e in exact:
    E = e["errors"]
    print(" & ".join([str(e["K"]), str(e["n"]), small(E["galerkin"]), sci(E["sdmd1"]), sci(E["sdmdR"]), sci(E["sdmd2"])]) + r" \\")
print("% Table 4, empirical matrices")
for e in empirical:
    M = e["mean"]
    print(" & ".join([str(e["K"]), str(e["n"]), small(M["exp"]), sci(M["sdmd1"]), sci(M["sdmdR"]), sci(M["sdmd2"])]) + r" \\")
print("% Table 5, rows for the torus")
for r in timing:
    print(" & ".join([str(r["K"]), str(r["N"]), plain(r["t_1R"]), plain(r["t_2"]), f"${r['t_2'] / r['t_1R']:.1f}$",
                      plain(r["ts"]["sdmdR"]), plain(r["t_exp"])]) + r" \\")

% Table 4, exact matrices
2 & 6 & $6.5\times10^{-4}$ & $4.20\times10^{-3}$ & $7.47\times10^{-4}$ & $6.82\times10^{-4}$ \\
4 & 22 & $2.4\times10^{-5}$ & $1.06\times10^{-3}$ & $3.05\times10^{-5}$ & $2.97\times10^{-5}$ \\
8 & 91 & $5.9\times10^{-8}$ & $2.53\times10^{-4}$ & $1.03\times10^{-6}$ & $1.03\times10^{-6}$ \\
16 & 391 & $<10^{-12}$ & $5.87\times10^{-5}$ & $5.52\times10^{-8}$ & $5.52\times10^{-8}$ \\
32 & 1692 & $<10^{-12}$ & $1.36\times10^{-5}$ & $2.94\times10^{-9}$ & $2.94\times10^{-9}$ \\
% Table 4, empirical matrices
2 & 6 & $6.6\times10^{-4}$ & $4.23\times10^{-3}$ & $7.43\times10^{-4}$ & $6.85\times10^{-4}$ \\
4 & 22 & $2.6\times10^{-5}$ & $1.06\times10^{-3}$ & $3.18\times10^{-5}$ & $3.00\times10^{-5}$ \\
8 & 91 & $9.6\times10^{-8}$ & $2.53\times10^{-4}$ & $1.04\times10^{-6}$ & $1.03\times10^{-6}$ \\
16 & 391 & $3.2\times10^{-12}$ & $5.87\times10^{-5}$ & $5.52\times10^{-8}$ & $5.52\times10^{-8}$ \\
% Table 5, rows for the torus
2 & 25 & $0.0069$ & $0.013$ & $1.8$ & $1.1\times1

## Software versions of this run

In [7]:
import os
import platform
import scipy

print("Python", platform.python_version(), "| NumPy", np.__version__, "| SciPy", scipy.__version__)
try:
    blas = np.show_config(mode="dicts")["Build Dependencies"]["blas"]
    print("BLAS:", blas.get("name"), blas.get("version"), "| logical cores:", os.cpu_count())
except Exception:
    print("logical cores:", os.cpu_count())

Python 3.13.18 | NumPy 2.5.3 | SciPy 1.18.1
BLAS: scipy-openblas 0.3.34.106.0 | logical cores: 2
